# Cotizador de exámenes médicos

Asistente que permite saber, en una sola consulta, **qué centros hacen el examen que se necesita,
cuánto cuesta y cuándo hay hora**. Informa, no decide.

Criterio de éxito (spec.md §1 y §7.1): el flujo corre de START a END; el reporte coincide con la
verdad del simulador; la trayectoria es visible; ante un límite el agente responde el límite.

**Flujo del grafo**

```text
START → router ─┬─► respuesta_directa ─────────────────────────────► END
                └─► agente ⇄ herramientas → consolidar → responder → verificador → END
```

Ejecute las celdas de arriba hacia abajo. Las celdas que dependen de red van en `try/except`.

## 1. Entorno
Verifica Python y que el `.env` tenga las credenciales (imprime solo si están presentes).

In [ ]:
import sys, os
from pathlib import Path
print("Python:", sys.version.split()[0])
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("Raíz:", RAIZ)
try:
    from dotenv import load_dotenv
    load_dotenv(RAIZ / ".env")
except Exception as exc:
    print("No se pudo cargar .env:", exc)
for var in ["OPENAI_API_KEY", "OPENAI_MODEL", "OPENAI_EMBEDDINGS_MODEL", "REDIS_URL"]:
    print(f"{var}: {'presente' if os.getenv(var) else 'AUSENTE'}")

## 2. Ficha del modelo y configuración
Se importa `config.py` (único lector del `.env`).

In [ ]:
try:
    import sys; sys.path.insert(0, str(RAIZ))
    import config
    for k, v in config.resumen().items():
        print(f"{k}: {v}")
except Exception as exc:
    print("Error importando config:", exc)

## 3. El estado y los modelos
Campo por campo de `state.py` y los modelos de `schemas.py`.

In [ ]:
try:
    import sys; sys.path.insert(0, str(RAIZ))
    import state, schemas
    print("Estado:")
    for campo in state.Estado.__annotations__:
        print(" -", campo)
    print("\nModelos de dominio:", [n for n in dir(schemas) if n[0].isupper()][:20])
except Exception as exc:
    print("Error:", exc)

## 4. El grafo
Construcción e impresión en mermaid.

In [ ]:
try:
    import sys; sys.path.insert(0, str(RAIZ))
    from main import app
    print(app.get_graph().draw_mermaid())
except Exception as exc:
    print("Error:", exc)

## 5. Datos de prueba
Documentos, instantáneas, eventos y escenarios (se cargan, no se generan).

In [ ]:
try:
    import json
    from pathlib import Path
    data = RAIZ / "data"
    print("PDFs:", sorted(p.name for p in (data / "documentos").glob("*.pdf")))
    print("Instantáneas:", sorted(p.name for p in (data / "instantaneas").glob("*.json")))
    print("Escenarios:", sorted(p.stem for p in (data / "escenarios").glob("*.json")))
    eventos = json.loads((data / "eventos.json").read_text(encoding="utf-8"))
    print("Eventos:", [e["id"] for e in eventos])
except Exception as exc:
    print("Error:", exc)

## 6. Ingesta y RAG
Extracción de exámenes por centro (sin tocar Redis).

In [ ]:
try:
    import sys; sys.path.insert(0, str(RAIZ))
    from services import ingesta
    examenes, fragmentos = ingesta.analizar_carpeta()
    print("Fragmentos:", len(fragmentos), "| Exámenes:", len(examenes))
    for e in examenes:
        print(f"  {e.centro_id} · {e.codigo} · {e.nombre}")
except Exception as exc:
    print("Error:", exc)

## 7. El simulador
Eventos del escenario `default` y `agenda`.

In [ ]:
try:
    import json
    from pathlib import Path
    for nombre in ["default", "agenda", "intenta_agendar"]:
        esc = json.loads((RAIZ / "data" / "escenarios" / f"{nombre}.json").read_text(encoding="utf-8"))
        print(nombre, "-> eventos:", {k: [e["evento"] for e in v] for k, v in esc["eventos"].items()})
except Exception as exc:
    print("Error:", exc)

## 8. Recorrido paso a paso
Caso ancla (requiere OpenAI y Redis; si faltan, se informa).

In [ ]:
try:
    import sys; sys.path.insert(0, str(RAIZ))
    from langchain_core.messages import HumanMessage
    from main import app
    thread = {"configurable": {"thread_id": "notebook-demo"}}
    for chunk in app.stream({"messages": [HumanMessage(content="Necesito la medición de glaucoma en Talca, Fonasa")],
                             "escenario_id": "default"}, thread, stream_mode="updates"):
        for nodo, update in chunk.items():
            print("NODO:", nodo)
    print("Respuesta:", app.get_state(thread).values["messages"][-1].content)
except Exception as exc:
    print("No se pudo ejecutar el recorrido (¿faltan credenciales/Redis?):", exc)

## 9. Los prompts
Contenido de cada prompt para estudiarlo.

In [ ]:
try:
    import sys; sys.path.insert(0, str(RAIZ))
    from prompts import seguridad, router_prompt, agente_prompt
    print("=== SEGURIDAD UNIVERSAL ==="); print(seguridad.BLOQUE_UNIVERSAL[:400])
    print("\n=== ROUTER (extracto) ==="); print(router_prompt.SYSTEM[:400])
except Exception as exc:
    print("Error:", exc)

## 10. Resultados de validación
Último informe del agente `validator`, si existe.

In [ ]:
try:
    from pathlib import Path
    xs = sorted((RAIZ / "validacion").glob("resultados_*.xlsx"))
    print("Informes:", [p.name for p in xs] or "ninguno todavía")
    if xs:
        from openpyxl import load_workbook
        wb = load_workbook(xs[-1]); print("Hojas:", wb.sheetnames)
except Exception as exc:
    print("Error:", exc)

## 11. Verificación final y zona de pruebas
Tabla de criterios y consulta libre.

In [ ]:
criterios = {
    "Flujo START→END": "req. verificación",
    "Reporte = verdad del simulador": "golden set",
    "Trazabilidad": "notebook + harness",
    "Tope de iteraciones": "main.despues_agente",
    "Seguridad S1–S6": "validacion/run_seguridad.py",
}
for k, v in criterios.items():
    print(f"✅ {k} — {v}")

# Zona de pruebas: escriba su propia consulta.
consulta = "Necesito un fondo de ojo en Talca, Fonasa"
try:
    import sys; sys.path.insert(0, str(RAIZ))
    from langchain_core.messages import HumanMessage
    from main import app
    thread = {"configurable": {"thread_id": "notebook-zona"}}
    for chunk in app.stream({"messages": [HumanMessage(content=consulta)], "escenario_id": "default"},
                            thread, stream_mode="updates"):
        print("NODO:", list(chunk.keys()))
    print(app.get_state(thread).values["messages"][-1].content)
except Exception as exc:
    print("Sin credenciales/Redis:", exc)